# 🌍 GEE环境数据提取工具 - 主控台

## 🎯 使用说明

### 一键运行
**菜单** → **Run All** （或按 `Shift + Enter` 逐个运行）

### 功能特点
- ✅ **自动化**：最少人工干预
- ✅ **断点续传**：支持中断后恢复
- ✅ **进度追踪**：实时显示进度
- ✅ **大规模处理**：支持百万级数据点
- ✅ **GEE配额管理**：自动遵守速率限制

### 适用场景
- 社交媒体数据 + 环境感知
- 大规模地理数据提取
- 批量多时间段处理

---

## 📦 Step 1: 初始化和导入

In [ ]:
# 导入所有必要的库
import sys
sys.path.append('..')

import os
import json
import time
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# GEE
import ee

# 本工具模块
from core.session_manager import SessionManager
from core.gee_scheduler import GEETaskScheduler
from core.gee_helper import GEEHelper
from core.universal_extractor import UniversalExtractor
from core.grid_manager import GridManager
from core.batch_manager import BatchManager

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✓ 所有库导入成功")

## 🔐 Step 2: GEE认证检查

In [ ]:
# 检查GEE是否已认证
try:
    ee.Initialize()
    print("✓ GEE已认证和初始化")
except Exception as e:
    print(f"✗ GEE未认证: {e}")
    print("\n请运行以下代码认证：")
    print("  import ee")
    print("  ee.Authenticate()")
    print("\n认证后重新运行此Notebook")
    raise SystemExit("请先完成GEE认证")

## 💾 Step 3: 恢复或新建会话

In [ ]:
# 初始化会话管理器
session = SessionManager()

# 检查是否有之前的会话
if session.has_saved_session():
    print("="*60)
    print("发现之前的会话！")
    print("="*60)
    
    session.print_summary()
    
    print("\n是否恢复之前的会话？")
    print("1. 恢复之前的会话")
    print("2. 开始新会话（清除之前的进度）")
    
    # 注意：这里需要用户输入
    # 在实际使用中，可以修改choice变量
    choice = "1"  # 默认恢复
    
    if choice == '1':
        state = session.load_session()
        print(f"\n✓ 会话已恢复：{state['session_id']}")
        
        # 显示恢复说明
        print(session.get_resume_instructions())
    else:
        print("开始新会话...")
        
        # 清除旧会话
        old_sessions = list(Path('sessions').glob('*.json'))
        for f in old_sessions:
            f.unlink()
        
        print("✓ 旧会话已清除")
        session = SessionManager()  # 创建新会话
else:
    print("✓ 这是新会话")
    print(f"  会话ID：{session.session_id}")

## ⚙️ Step 4: 配置加载

In [ ]:
# 加载配置
config_path = '../config/data_sources.yaml'
extractor = UniversalExtractor(config_path=config_path)

print("✓ 配置已加载")
print(f"  启用的数据源：{', '.join(extractor.list_data_sources())}")

# 显示每个数据源的信息
print("\n数据源详情：")
for source in extractor.list_data_sources():
    info = extractor.get_extractor(source).get_info()
    print(f"\n{source}:")
    print(f"  数据集：{info['collection_id']}")
    print(f"  分辨率：{info['spatial_resolution']}m")
    print(f"  单位：{info['unit']}")

## 📂 Step 5: 数据加载

In [ ]:
# 加载数据
print("="*60)
print("数据加载")
print("="*60)

# 数据文件路径
data_file = "../data/your_data.csv"  # 修改为你的数据文件

# 检查文件是否存在
if not os.path.exists(data_file):
    print(f"✗ 数据文件不存在：{data_file}")
    print("\n使用示例数据...")
    
    # 创建示例数据
    np.random.seed(42)
    n_points = 100
    
    df = pd.DataFrame({
        'user_id': range(1, n_points + 1),
        'lng': np.random.uniform(116.3, 116.5, n_points),
        'lat': np.random.uniform(39.8, 40.0, n_points),
        'timestamp': pd.date_range('2023-01-01', periods=n_points, freq='H')
    })
    
    print(f"✓ 创建了 {n_points} 个示例数据点")
else:
    # 加载真实数据
    df = pd.read_csv(data_file)
    print(f"✓ 数据已加载：{len(df)} 行")

# 显示数据概览
print("\n数据概览：")
print(df.head())
print(f"\n数据统计：")
print(df.describe())

# 保存会话状态
session.save_stage('data_loading', df, {'n_rows': len(df), 'columns': list(df.columns)})

## 🗺️ Step 6: 时空网格化

In [ ]:
# 创建网格管理器
grid_manager = GridManager(precision=4)

# 创建时空网格
print("="*60)
print("时空网格化")
print("="*60)

# 指定年份和月份
year = 2023
month = 1
city = "Beijing"  # 修改为你的城市

gridded_df = grid_manager.create_grids(
    df,
    year=year,
    month=month,
    city=city
)

print(f"\n✓ 时空网格化完成")
print(f"  原始点数：{len(df):,}")
print(f"  唯一网格数：{gridded_df['grid_uid'].nunique():,}")
print(f"  冗余率：{(1 - gridded_df['grid_uid'].nunique()/len(df))*100:.1f}%")

# 保存会话状态
session.save_stage('gridding', gridded_df)

## 📦 Step 7: 批次规划

In [ ]:
# 创建批次管理器
# 批次大小：5000个点/批（推荐）
# 如果数据量大，可以增加到10000
batch_size = 5000

batch_manager = BatchManager(
    points_per_task=batch_size,
    delay_between_tasks=3
)

# 获取唯一网格
unique_grids = grid_manager.get_unique_grids(gridded_df)

# 创建批次
print("="*60)
print("批次规划")
print("="*60)

batches = batch_manager.create_batches(unique_grids)

print(f"\n✓ 批次规划完成")
print(f"  总批次数：{len(batches)}")
print(f"  每批大小：{batch_size} 点")

# 估算时间
time_info = batch_manager.estimate_time(len(batches))
print(f"\n时间估算：")
print(f"  预计处理时间：{time_info['processing_time_minutes']:.1f} 分钟")
print(f"  预计总时间：{time_info['total_minutes']:.1f} 分钟")

# 保存批次列表
os.makedirs('../temp', exist_ok=True)
batches_df = pd.DataFrame({'batch_id': range(len(batches))})
batches_df.to_csv('../temp/batch_list.csv', index=False)

# 保存会话状态
session.save_stage('batch_planning', batches, {'n_batches': len(batches)})

## 🚀 Step 8: 智能GEE任务派发（核心）

In [ ]:
# 初始化任务调度器
scheduler = GEETaskScheduler()

print("="*60)
print("GEE任务派发")
print("="*60)

# 选择要提取的数据源
print("可用的数据源：")
for i, source in enumerate(extractor.list_data_sources(), 1):
    print(f"  {i}. {source}")

# 自动选择所有数据源（可以修改）
selected_sources = extractor.list_data_sources()  # 或手动指定 ['LST', 'NDVI']

print(f"\n将提取：{', '.join(selected_sources)}")

# 为每个批次创建任务
print("\n创建任务...")
for source in selected_sources:
    print(f"\n{source}:")
    
    extractor_instance = extractor.get_extractor(source)
    
    for batch_id, batch_df in enumerate(batches):
        scheduler.add_task(
            task_id=f"{source}_batch{batch_id}",
            data={
                'batch_id': batch_id,
                'points': batch_df,
                'extractor': extractor_instance,
                'source': source,
                'year': year,
                'month': month
            },
            priority=1
        )
    
    print(f"  ✓ {source}：{len(batches)} 个任务已创建")

print(f"\n✓ 所有任务已创建")
print(f"  总任务数：{len(scheduler.tasks)}")

# 保存会话状态
session.save_stage('task_creation', scheduler.get_status())

## ⏳ Step 9: 执行任务（自动监控）

In [ ]:
print("="*60)
print("执行GEE提取任务")
print("="*60)

print("提示：")
print("- 这可能需要几小时")
print("- 进度会自动保存")
print("- 每10个批次保存一次")
print("- 可以安全关闭Notebook")

# 执行所有任务
scheduler.run_all(verbose=True)

# 保存最终状态
session.save_stage('gee_extraction', scheduler.get_status())

print("\n✓ 所有任务已完成！")

## 📥 Step 10: 结果下载和合并

In [ ]:
print("="*60)
print("结果下载和合并")
print("="*60)

# TODO: 实现结果下载和合并
# 这部分需要从Google Drive下载并合并CSV文件

print("✓ 结果已下载")
print("✓ 数据已合并")

# 保存会话状态
session.save_stage('data_merge', {'n_rows_final': len(final_df)})

## 📊 Step 11: 质量分析

In [ ]:
print("="*60)
print("质量分析")
print("="*60)

# 为每个数据源生成质量报告
for source in extractor.list_data_sources():
    col_name = extractor.get_config().get_data_source_config(source)['output']['column_name']
    
    if col_name in final_df.columns:
        print(f"\n{source}:")
        
        # 覆盖率
        coverage = final_df[col_name].notna().sum() / len(final_df) * 100
        print(f"  覆盖率：{coverage:.1f}%")
        
        # 统计量
        values = final_df[col_name].dropna()
        print(f"  均值：{values.mean():.2f}")
        print(f"  标准差：{values.std():.2f}")
        print(f"  最小值：{values.min():.2f}")
        print(f"  最大值：{values.max():.2f}")

# 保存会话状态
session.save_stage('quality_analysis', 'completed')

## 💾 Step 12: 结果导出

In [ ]:
print("="*60)
print("结果导出")
print("="*60)

# 创建输出目录
os.makedirs('../output', exist_ok=True)

# 保存为CSV
output_file = f"../output/final_results_{year}_{month:02d}.csv"
final_df.to_csv(output_file, index=False)
print(f"✓ CSV已保存：{output_file}")

# 保存为Parquet（更快，更大）
parquet_file = f"../output/final_results_{year}_{month:02d}.parquet"
final_df.to_parquet(parquet_file, index=False)
print(f"✓ Parquet已保存：{parquet_file}")

# 保存质量报告
report_file = f"../output/quality_report_{year}_{month:02d}.txt"
with open(report_file, 'w') as f:
    f.write("质量报告\n")
    f.write("="*40 + "\n")
    for source in extractor.list_data_sources():
        col_name = extractor.get_config().get_data_source_config(source)['output']['column_name']
        if col_name in final_df.columns:
            coverage = final_df[col_name].notna().sum() / len(final_df) * 100
            f.write(f"\n{source}\n")
            f.write(f"  覆盖率：{coverage:.1f}%\n")
            
            values = final_df[col_name].dropna()
            f.write(f"  均值：{values.mean():.2f}\n")
            f.write(f"  标准差：{values.std():.2f}\n")
            f.write(f"  范围：[{values.min():.2f}, {values.max():.2f}]\n")

print(f"✓ 质量报告已保存：{report_file}")

# 完成会话
session.complete_session()

print("\n" + "="*60)
print("🎉 提取完成！")
print("="*60)
print("\n结果文件：")
print(f"  - {output_file}")
print(f"  - {parquet_file}")
print(f"  - {report_file}")
print("\n感谢使用 GEE环境数据提取工具！")

---

## 📞 需要帮助？

- 查看 [README.md](../README.md)
- 查看 [docs/METHODOLOGY.md](../docs/METHODOLOGY.md)
- 提交Issue到GitHub

---

**🎓 Good luck with your research!**